# TRIBE v2 — Backend on Colab
Runs the FastAPI backend + model on Colab's GPU.

**Run cells top to bottom. After cell 3, restart the runtime, then continue from cell 4.**

In [ ]:
# Step 1: System deps
!apt-get install -y -q ffmpeg
print('ffmpeg ready')

In [ ]:
# Step 2: Clone tribev2 repo (needed before pip install)
import os
if not os.path.exists('/content/tribev2'):
    !git clone -q https://github.com/facebookresearch/tribev2 /content/tribev2
else:
    print('tribev2 already cloned')

In [ ]:
# Step 3: Install ALL dependencies in correct order
# This resolves all version conflicts. RESTART RUNTIME after this cell.

# 1. Upgrade torch to satisfy whisperx + pyannote requirements
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. Install tribev2 (pins numpy==2.2.6 and other strict deps)
!pip install -q -e /content/tribev2

# 3. Install whisperx AFTER torch upgrade
!pip install -q whisperx

# 4. Force numpy to exactly what tribev2 requires (last writer wins)
!pip install -q numpy==2.2.6 --force-reinstall

# 5. Remaining backend deps
!pip install -q fastapi 'uvicorn[standard]' yt-dlp pydantic nest_asyncio huggingface_hub

print('\n=== All packages installed ===')
print('>>> RESTART RUNTIME NOW (Runtime > Restart runtime), then continue from Step 4 <<<')

In [ ]:
# Step 4 (after runtime restart): Check GPU + verify installs
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import numpy as np
print(f'numpy: {np.__version__}')
print(f'torch: {torch.__version__}')

import tribev2, inspect, pkgutil, importlib
print(f'tribev2 loaded from: {tribev2.__file__}')

# Auto-discover the model class
ModelClass = None
for _, obj in inspect.getmembers(tribev2, inspect.isclass):
    if hasattr(obj, 'from_pretrained'):
        ModelClass = obj
        break
if ModelClass is None:
    for _, modname, _ in pkgutil.walk_packages(tribev2.__path__, prefix='tribev2.'):
        try:
            mod = importlib.import_module(modname)
            for _, obj in inspect.getmembers(mod, inspect.isclass):
                if hasattr(obj, 'from_pretrained'):
                    ModelClass = obj
                    break
        except Exception:
            pass
        if ModelClass:
            break

if ModelClass:
    print(f'Model class found: {ModelClass.__module__}.{ModelClass.__name__}')
else:
    print('ERROR: No model class found in tribev2!')

In [ ]:
# Step 5: Clone/update brain-neuro repo
import os
if os.path.exists('/content/brain-neuro'):
    !git -C /content/brain-neuro pull -q
    print('brain-neuro updated')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git /content/brain-neuro
    print('brain-neuro cloned')

os.chdir('/content/brain-neuro')
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Step 6: Download model weights from HuggingFace
from google.colab import userdata
import huggingface_hub

try:
    hf_token = userdata.get('HF_TOKEN')
    print('Using HF_TOKEN from Colab secrets')
except Exception:
    hf_token = input('Enter your HuggingFace token: ')

huggingface_hub.login(token=hf_token, add_to_git_credential=False)
huggingface_hub.hf_hub_download(
    repo_id='facebook/tribev2',
    filename='best.ckpt',
    local_dir='model_weights'
)
print('Model weights downloaded.')

In [ ]:
# Step 7: Patch config to use GPU if available
import re, torch

cfg_path = 'model_weights/config.yaml'
with open(cfg_path) as f:
    cfg = f.read()

target_device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = re.sub(r'device: (cpu|cuda)', f'device: {target_device}', cfg)

with open(cfg_path, 'w') as f:
    f.write(cfg)

print(f'Config patched: device → {target_device}')

In [ ]:
# Step 8 (Optional): Upload cookies.txt for Instagram/TikTok
# Skip this cell if you only need YouTube.
from google.colab import files
import shutil

print('Upload cookies.txt (or skip for YouTube-only)')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, 'cookies.txt')
    print('cookies.txt saved')

In [ ]:
# Step 9: Install cloudflared tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ready')

In [ ]:
# Step 10: Start backend + tunnel
import nest_asyncio, uvicorn, threading, subprocess, time, re as _re

nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)
print('FastAPI started on port 8000')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('Waiting for tunnel URL...')
for line in proc.stdout:
    line = line.decode()
    match = _re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group()
        print(f'\n=========================================')
        print(f'PUBLIC BACKEND URL:')
        print(f'  {public_url}')
        print(f'=========================================')
        print(f'Paste this into the Backend field on your frontend.')
        break